# 3. AG News Classification with Pretrained GloVe Embeddings

**New compared to notebook 2:** the embedding layer is initialised with pretrained [GloVe](https://nlp.stanford.edu/projects/glove/) vectors (100 dimensions, trained on Wikipedia + Gigaword, 400,000 words) and **frozen**. The vocabulary is now GloVe's, so far fewer words are unknown, and the LSTM starts from embeddings that already encode word meaning instead of learning them from 120,000 articles. The data pipeline, the LSTM and the training loop are unchanged.

Get the vectors from https://nlp.stanford.edu/projects/glove/ (`glove.6B.zip`, 822 MB) and put `glove.6B.100d.txt` in `models/`.

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
# Hyperparameters
max_len = 64
vocab_size = None
dim = 100
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
batch_size = 32
learning_rate = 1e-3
epochs = 5
seed = 2024

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True

## Load data

In [ ]:
from utils import get_data
X_train, y_train, X_test, y_test = get_data()

## Split data into words, calculate length of each sentence,
## and calculate the statistics of the dataset

In [ ]:
lens = []
for sentence in X_train:
    lens.append(len(sentence.split(' ')))
lens = np.array(lens)
lens.mean(), np.percentile(lens, 95), lens.max()

## Load Embeddings

In [ ]:
file_name = 'models/glove.6B.100d.txt'
# id 0 = padding, id 1 = unknown word (both zero vectors); GloVe words from 2
word2idx, idx2word = {'<pad>': 0, '<unk>': 1}, {0: '<pad>', 1: '<unk>'}
word2vec = [np.zeros(dim), np.zeros(dim)]
with open(file_name, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        word, vec = line.split(' ', 1)
        vec = np.fromstring(vec, sep=' ')
        word2idx[word] = idx + 2
        idx2word[idx + 2] = word
        word2vec.append(vec)
word2vec = np.stack(word2vec)
vocab_size = word2vec.shape[0]
word2vec.shape

## Preprocess data and create the tensors

In [ ]:
# prepare the data: words -> ids; unknown words -> <unk> (id 1)
def tokenize(sentences):
    out = []
    for sentence in sentences:
        sentence = sentence.split(' ')[:max_len]
        sentence = [word2idx.get(word, 1) for word in sentence]
        sentence = [0] * (max_len - len(sentence)) + sentence  # prepad the sentence
        out.append(sentence)
    return torch.tensor(out)

X_train_tensor = tokenize(X_train)
y_train_tensor = torch.tensor(y_train)
X_test_tensor = tokenize(X_test)
y_test_tensor = torch.tensor(y_test)

In [ ]:
# data loader
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
# train loader in train loop

test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Create Model

In [ ]:
class Model(nn.Module):
    def __init__(self, vocab_size, hid_dim, out_dim):
        super(Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hid_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(word2vec).float())
        self.embedding.weight.requires_grad = False
        self.rnn = nn.LSTM(hid_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, out_dim)
    
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

In [ ]:
model = Model(vocab_size, dim, 4).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Train

In [ ]:
for epoch in range(epochs):
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True)
    model.train()
    for X_batch, y_batch in tqdm(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        logits = model(X_batch)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.nll_loss(log_probs, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            y_pred = F.log_softmax(y_pred, dim=1)
            _, predicted = torch.max(y_pred, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        acc = correct / total
    print(f'Epoch: {epoch+1:02d}, Test Acc: {acc:.4f}')